# Whiskey Tariffs in Iowa: Prices, Sales and Availability

What happened to imported whiskey and brandy in Iowa after the 2025 US tariffs on UK and EU spirits?

**Short answer:** Iowa's wholesale prices for imported products did not move, but imported sales still fell 7 to 10 percentage points more than American whiskey. The decline came through a narrower product range (EU) and fewer stores ordering (UK), not through price.

See the README for the full write-up. This notebook runs every step in order.

## Setup

Data: Iowa Liquor Sales for 2024, 2025 and 2026 from [data.iowa.gov](https://data.iowa.gov) (search "Iowa Liquor Sales"). Each year downloads as a zip of CSV parts. Unzip them into one folder and point `DATA_DIR` at it.

The notebook was built in Google Colab with the data in Google Drive. To run it elsewhere, skip the Drive lines and change `DATA_DIR`.

In [ ]:
!pip install -q duckdb

from google.colab import drive
drive.mount('/content/drive')

import os, duckdb
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_DIR = '/content/drive/MyDrive/tariff-project'
os.chdir(DATA_DIR)
con = duckdb.connect('tariff.duckdb')

# Read only the original Iowa files (not any summary CSVs saved in the same folder).
# The price columns are blank before July 2025, so DuckDB reads them as text: convert them to numbers.
con.sql(f"""
    CREATE OR REPLACE VIEW sales AS
    SELECT * REPLACE (
        TRY_CAST(state_bottle_cost   AS DOUBLE) AS state_bottle_cost,
        TRY_CAST(state_bottle_retail AS DOUBLE) AS state_bottle_retail
    )
    FROM read_csv_auto('{DATA_DIR}/**/iowa_liquor_sales_*.csv',
                       union_by_name = true, filename = true)
""")
print("Ready")

## Step 1: What is in the data?

8.1 million sales from January 2024 to October 30, 2026. Every row is a licensed store buying from the state of Iowa, which is the wholesaler for spirits.

In [ ]:
con.sql("""
    SELECT YEAR(ordered_on) AS year,
           COUNT(*) AS row_count,
           MIN(ordered_on) AS first_date,
           MAX(ordered_on) AS last_date
    FROM sales
    GROUP BY year
    ORDER BY year
""").df()

## Step 2: Can the price columns be trusted?

`state_bottle_cost` = what Iowa pays the supplier or importer. `state_bottle_retail` = what Iowa charges stores (cost plus its markup).

**Finding:** both columns are blank from January 2024 to June 2025 and only filled from July 2025, which is after the first tariff.

In [ ]:
con.sql("""
    SELECT DATE_TRUNC('month', ordered_on) AS month,
           COUNT(*) AS row_count,
           ROUND(100.0 * COUNT(state_bottle_cost)   / COUNT(*), 1) AS pct_cost_filled,
           ROUND(100.0 * COUNT(state_bottle_retail) / COUNT(*), 1) AS pct_retail_filled
    FROM sales
    GROUP BY month
    ORDER BY month
""").df()

Can the missing prices be rebuilt as `sales_dollars / sales_bottles`? It matches the official list price 80.6% of the time. Iowa's markup is typically 50% (retail = 1.5 x cost).

In [ ]:
con.sql("""
    SELECT
        ROUND(100.0 * AVG(CASE WHEN ABS(sales_dollars / sales_bottles - state_bottle_retail) < 0.01
                              THEN 1 ELSE 0 END), 1) AS pct_price_matches_retail,
        ROUND(MEDIAN(state_bottle_retail / state_bottle_cost), 3) AS typical_markup_ratio,
        ROUND(MIN(state_bottle_retail / state_bottle_cost), 3) AS min_ratio,
        ROUND(MAX(state_bottle_retail / state_bottle_cost), 3) AS max_ratio
    FROM sales
    WHERE state_bottle_cost > 0
      AND sales_bottles > 0
""").df()

How do the mismatches differ? Discounts are rare (under 1%). About 18.6% of sales are roughly 3% above the list price.

In [ ]:
con.sql("""
    SELECT
        CASE
            WHEN ABS(sales_dollars / sales_bottles - state_bottle_retail) < 0.01 THEN '1. exact match'
            WHEN sales_dollars / sales_bottles < state_bottle_retail THEN '2. paid LESS than retail'
            ELSE '3. paid MORE than retail'
        END AS comparison,
        COUNT(*) AS row_count,
        ROUND(MEDIAN((sales_dollars / sales_bottles) / state_bottle_retail), 3) AS typical_ratio
    FROM sales
    WHERE state_bottle_retail > 0
      AND sales_bottles > 0
    GROUP BY comparison
    ORDER BY comparison
""").df()

The extra amount is a flat $0.50 per bottle across almost every bottle size, so it is a fixed charge, not a price difference.

In [ ]:
con.sql("""
    SELECT bottle_volume_ml,
           COUNT(*) AS row_count,
           ROUND(MEDIAN(sales_dollars / sales_bottles - state_bottle_retail), 2) AS median_extra_per_bottle
    FROM sales
    WHERE state_bottle_retail > 0
      AND sales_bottles > 0
      AND sales_dollars / sales_bottles - state_bottle_retail >= 0.01
    GROUP BY bottle_volume_ml
    ORDER BY row_count DESC
    LIMIT 10
""").df()

Is the $0.50 a broken-case fee? No: full-case orders pay it more often (22.3%) than broken cases (12.0%). Cause unknown, so it is handled rather than explained.

In [ ]:
con.sql("""
    SELECT
        CASE WHEN sales_bottles % pack = 0 THEN 'full cases' ELSE 'broken case' END AS order_type,
        COUNT(*) AS row_count,
        ROUND(100.0 * AVG(CASE WHEN sales_dollars / sales_bottles - state_bottle_retail >= 0.01
                              THEN 1 ELSE 0 END), 1) AS pct_paid_extra
    FROM sales
    WHERE state_bottle_retail > 0
      AND sales_bottles > 0
      AND pack > 0
    GROUP BY order_type
""").df()

Does a typical price per product per month fix it? The median matches the list price for 87.9% of product-months, the most common price (mode) for 88.2%, but only 77.8% when weighted by bottles sold.

In [ ]:
con.sql("""
    WITH product_month AS (
        SELECT item_no,
               DATE_TRUNC('month', ordered_on) AS month,
               MEDIAN(sales_dollars / sales_bottles) AS median_price,
               MODE(ROUND(sales_dollars / sales_bottles, 2)) AS mode_price,
               MEDIAN(state_bottle_retail) AS list_price,
               SUM(sales_bottles) AS bottles
        FROM sales
        WHERE state_bottle_retail > 0
          AND sales_bottles > 0
        GROUP BY item_no, month
    )
    SELECT
        ROUND(100.0 * AVG(CASE WHEN ABS(median_price - list_price) / list_price < 0.01 THEN 1 ELSE 0 END), 1) AS median_within_1pct,
        ROUND(100.0 * AVG(CASE WHEN ABS(mode_price - list_price) / list_price < 0.01 THEN 1 ELSE 0 END), 1) AS mode_within_1pct,
        ROUND(100.0 * SUM(CASE WHEN ABS(mode_price - list_price) / list_price < 0.01 THEN bottles ELSE 0 END)
              / SUM(bottles), 1) AS mode_within_1pct_by_bottles
    FROM product_month
""").df()

The project measures price **changes**, so the real test is whether the rebuilt price moves when the list price moves. It agrees 88.3% of the time (85.7% by bottles), below the 95% bar set before running the test.

**Decision:** the main price analysis uses official prices from July 2025 onward. Bottles sold (complete for all years) carries the before-and-after comparison. Rebuilt prices before July 2025 are rough supporting evidence only.

In [ ]:
con.sql("""
    WITH product_month AS (
        SELECT item_no,
               DATE_TRUNC('month', ordered_on) AS month,
               MODE(ROUND(sales_dollars / sales_bottles, 2)) AS calc_price,
               MEDIAN(state_bottle_retail) AS list_price,
               SUM(sales_bottles) AS bottles
        FROM sales
        WHERE state_bottle_retail > 0
          AND sales_bottles > 0
        GROUP BY item_no, month
    ),
    changes AS (
        SELECT item_no, month, bottles,
               calc_price / LAG(calc_price) OVER (PARTITION BY item_no ORDER BY month) - 1 AS calc_change,
               list_price / LAG(list_price) OVER (PARTITION BY item_no ORDER BY month) - 1 AS list_change
        FROM product_month
    )
    SELECT COUNT(*) AS month_to_month_pairs,
           ROUND(100.0 * AVG(CASE WHEN ABS(calc_change - list_change) < 0.01 THEN 1 ELSE 0 END), 1) AS pct_changes_agree,
           ROUND(100.0 * SUM(CASE WHEN ABS(calc_change - list_change) < 0.01 THEN bottles ELSE 0 END)
                 / SUM(bottles), 1) AS pct_agree_by_bottles
    FROM changes
    WHERE calc_change IS NOT NULL
""").df()

## Step 3: Build the groups

Find the whiskey and brandy categories, check what the unclear ones contain, then sort each category into UK, EU or American (the comparison group).

In [ ]:
con.sql("""
    SELECT category_name,
           COUNT(*) AS row_count,
           SUM(sales_bottles) AS bottles
    FROM sales
    WHERE category_name ILIKE '%whisk%'
       OR category_name ILIKE '%scotch%'
       OR category_name ILIKE '%bourbon%'
       OR category_name ILIKE '%brand%'
       OR category_name ILIKE '%cognac%'
    GROUP BY category_name
    ORDER BY bottles DESC
""").df()

In [ ]:
# Top 5 products in the categories whose origin is unclear from the name
con.sql("""
    SELECT category_name,
           im_desc AS product,
           vendor_name,
           SUM(sales_bottles) AS bottles
    FROM sales
    WHERE category_name IN ('IMPORTED BRANDIES', 'BLENDED WHISKIES',
                            'WHISKEY LIQUEUR', 'CANADIAN WHISKIES')
    GROUP BY category_name, im_desc, vendor_name
    QUALIFY ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY bottles DESC) <= 5
    ORDER BY category_name, bottles DESC
""").df()

Imported Brandies is mostly French cognac (Hennessy, Remy Martin), so it goes in the EU group. Blended Whiskies are American blends. Canadian whiskies (a separate tariff story) and flavored whiskey liqueurs are left out.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE category_groups AS
    SELECT * FROM (VALUES
        ('SCOTCH WHISKIES',                'UK'),
        ('SINGLE MALT SCOTCH',             'UK'),
        ('IRISH WHISKIES',                 'EU'),
        ('IMPORTED BRANDIES',              'EU'),
        ('STRAIGHT BOURBON WHISKIES',      'American'),
        ('TENNESSEE WHISKIES',             'American'),
        ('STRAIGHT RYE WHISKIES',          'American'),
        ('SINGLE BARREL BOURBON WHISKIES', 'American'),
        ('BOTTLED IN BOND BOURBON',        'American'),
        ('CORN WHISKIES',                  'American'),
        ('AMERICAN BRANDIES',              'American'),
        ('BLENDED WHISKIES',               'American')
    ) AS t(category_name, origin)
""")

One row per product per month: 8.1 million rows become about 31,000.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE whiskey_monthly AS
    SELECT
        DATE_TRUNC('month', s.ordered_on) AS month,
        g.origin,
        s.category_name,
        s.item_no,
        s.im_desc AS product,
        s.vendor_name,
        s.bottle_volume_ml,
        SUM(s.sales_bottles) AS bottles,
        SUM(s.sales_dollars) AS dollars,
        SUM(s.sales_liters)  AS liters,
        MEDIAN(s.state_bottle_cost)   AS supplier_cost,
        MEDIAN(s.state_bottle_retail) AS list_price,
        MODE(ROUND(s.sales_dollars / NULLIF(s.sales_bottles, 0), 2)) AS calc_price
    FROM sales s
    JOIN category_groups g
      ON s.category_name = g.category_name
    GROUP BY 1, 2, 3, 4, 5, 6, 7
""")

con.sql("SELECT COUNT(*) AS row_count FROM whiskey_monthly").df()

In [ ]:
# Sample size, by group and in total (the blank origin row is the total)
con.sql("""
    SELECT g.origin,
           COUNT(*) AS sales_records,
           COUNT(DISTINCT s.store_no) AS stores,
           COUNT(DISTINCT s.item_no) AS products,
           ROUND(SUM(s.sales_bottles)) AS bottles
    FROM sales s
    JOIN category_groups g ON s.category_name = g.category_name
    GROUP BY ROLLUP (g.origin)
    ORDER BY g.origin NULLS LAST
""").df()

In [ ]:
# Save the monthly table for Power BI
con.sql("COPY whiskey_monthly TO 'whiskey_monthly.csv' (HEADER)")

## Step 4: Did imported sales fall more than American?

Bottles sold by group, each indexed to its own 2024 monthly average. In 2024, before any tariff, the three groups move together, which supports American whiskey as a fair comparison group.

In [ ]:
df = con.sql("""
    WITH monthly AS (
        SELECT month, origin, SUM(bottles) AS bottles
        FROM whiskey_monthly
        GROUP BY month, origin
    ),
    baseline AS (
        SELECT origin, AVG(bottles) AS avg_2024
        FROM monthly
        WHERE YEAR(month) = 2024
        GROUP BY origin
    )
    SELECT m.month, m.origin,
           ROUND(100.0 * m.bottles / b.avg_2024, 1) AS index_2024
    FROM monthly m
    JOIN baseline b ON m.origin = b.origin
    ORDER BY m.month, m.origin
""").df()

chart = df.pivot(index='month', columns='origin', values='index_2024')
ax = chart.plot(figsize=(12, 5), linewidth=2)

events = [('2025-04-01', '10% tariff'), ('2025-08-01', 'EU raised to 15%'), ('2026-07-01', 'UK exempted')]
for date, label in events:
    ax.axvline(pd.Timestamp(date), color='gray', linestyle='--')
    ax.text(pd.Timestamp(date), ax.get_ylim()[1], ' ' + label, rotation=90, va='top', fontsize=9)

ax.set_title('Bottles sold by origin (2024 monthly average = 100)')
ax.set_ylabel('Index')
ax.set_xlabel('')
plt.show()

Compare the same months each year (April to October) to avoid the holiday spike. The gap = import group's change minus American's change.

In [ ]:
con.sql("""
    WITH tagged AS (
        SELECT origin, bottles,
               CASE
                   WHEN month BETWEEN '2024-04-01' AND '2024-10-01' THEN 'y2024'
                   WHEN month BETWEEN '2025-04-01' AND '2025-10-01' THEN 'y2025'
                   WHEN month BETWEEN '2026-04-01' AND '2026-10-01' THEN 'y2026'
               END AS period
        FROM whiskey_monthly
    ),
    totals AS (
        SELECT origin,
               SUM(CASE WHEN period = 'y2024' THEN bottles END) AS apr_oct_2024,
               SUM(CASE WHEN period = 'y2025' THEN bottles END) AS apr_oct_2025,
               SUM(CASE WHEN period = 'y2026' THEN bottles END) AS apr_oct_2026
        FROM tagged
        GROUP BY origin
    )
    SELECT origin,
           ROUND(apr_oct_2024) AS apr_oct_2024,
           ROUND(apr_oct_2025) AS apr_oct_2025,
           ROUND(apr_oct_2026) AS apr_oct_2026,
           ROUND(100.0 * (apr_oct_2025 / apr_oct_2024 - 1), 1) AS pct_change_2025,
           ROUND(100.0 * (apr_oct_2026 / apr_oct_2024 - 1), 1) AS pct_change_2026
    FROM totals
    ORDER BY origin
""").df()

## Step 5: Did import prices rise?

Same product against itself (so a shift toward cheaper bottles cannot fake a price change), official supplier price, July 2025 = 100.

In [ ]:
price = con.sql("""
    WITH base AS (
        SELECT item_no, MEDIAN(supplier_cost) AS base_cost
        FROM whiskey_monthly
        WHERE month = '2025-07-01' AND supplier_cost > 0
        GROUP BY item_no
    ),
    product_index AS (
        SELECT w.month, w.origin, w.item_no, w.bottles,
               w.supplier_cost / b.base_cost AS cost_ratio
        FROM whiskey_monthly w
        JOIN base b ON w.item_no = b.item_no
        WHERE w.month >= '2025-07-01'
          AND w.supplier_cost > 0
          AND w.bottles > 0
    )
    SELECT month, origin,
           COUNT(*) AS products,
           ROUND(100 * MEDIAN(cost_ratio), 1) AS median_index,
           ROUND(100 * SUM(cost_ratio * bottles) / SUM(bottles), 1) AS weighted_index
    FROM product_index
    GROUP BY month, origin
    ORDER BY month, origin
""").df()

chart = price.pivot(index='month', columns='origin', values='weighted_index')
ax = chart.plot(figsize=(12, 5), linewidth=2, marker='o')
for date, label in [('2025-08-01', 'EU raised to 15%'), ('2026-07-01', 'UK exempted')]:
    ax.axvline(pd.Timestamp(date), color='gray', linestyle='--')
    ax.text(pd.Timestamp(date), ax.get_ylim()[1], ' ' + label, rotation=90, va='top', fontsize=9)
ax.axhline(100, color='black', linewidth=0.8)
ax.set_title('Supplier price of the same products (July 2025 = 100, weighted by bottles sold)')
ax.set_ylabel('Index')
ax.set_xlabel('')
plt.show()

price

Share of products whose supplier price went up, stayed the same or went down, July to October 2025 vs July to October 2026.

In [ ]:
con.sql("""
    WITH per_product AS (
        SELECT origin, item_no,
               MEDIAN(CASE WHEN month BETWEEN '2025-07-01' AND '2025-10-01' THEN supplier_cost END) AS cost_2025,
               MEDIAN(CASE WHEN month BETWEEN '2026-07-01' AND '2026-10-01' THEN supplier_cost END) AS cost_2026
        FROM whiskey_monthly
        GROUP BY origin, item_no
    )
    SELECT origin,
           COUNT(*) AS products,
           ROUND(100.0 * AVG(CASE WHEN cost_2026 > cost_2025 * 1.005 THEN 1 ELSE 0 END), 1) AS pct_went_up,
           ROUND(100.0 * AVG(CASE WHEN ABS(cost_2026 / cost_2025 - 1) <= 0.005 THEN 1 ELSE 0 END), 1) AS pct_no_change,
           ROUND(100.0 * AVG(CASE WHEN cost_2026 < cost_2025 * 0.995 THEN 1 ELSE 0 END), 1) AS pct_went_down
    FROM per_product
    WHERE cost_2025 > 0 AND cost_2026 > 0
    GROUP BY origin
    ORDER BY origin
""").df()

Before July 2025: rough rebuilt price, by quarter, 750ml and larger bottles only (where the $0.50 charge is a small share of the price), Q1 2024 = 100. No jump after the April 2025 tariff.

In [ ]:
rough = con.sql("""
    WITH quarterly AS (
        SELECT DATE_TRUNC('quarter', month) AS quarter,
               origin, item_no,
               MEDIAN(calc_price) AS price,
               SUM(bottles) AS bottles
        FROM whiskey_monthly
        WHERE bottle_volume_ml >= 750
          AND calc_price > 0
          AND bottles > 0
        GROUP BY quarter, origin, item_no
    ),
    base AS (
        SELECT item_no, price AS base_price
        FROM quarterly
        WHERE quarter = '2024-01-01'
    )
    SELECT q.quarter, q.origin,
           COUNT(*) AS products,
           ROUND(100 * MEDIAN(q.price / b.base_price), 1) AS median_index,
           ROUND(100 * SUM(q.price / b.base_price * q.bottles) / SUM(q.bottles), 1) AS weighted_index
    FROM quarterly q
    JOIN base b ON q.item_no = b.item_no
    GROUP BY q.quarter, q.origin
    ORDER BY q.quarter, q.origin
""").df()

chart = rough.pivot(index='quarter', columns='origin', values='weighted_index')
ax = chart.plot(figsize=(12, 5), linewidth=2, marker='o')
ax.axvline(pd.Timestamp('2025-04-01'), color='gray', linestyle='--')
ax.text(pd.Timestamp('2025-04-01'), ax.get_ylim()[1], ' 10% tariff', rotation=90, va='top', fontsize=9)
ax.axhline(100, color='black', linewidth=0.8)
ax.set_title('Rough price of the same products (Q1 2024 = 100, 750ml and larger)')
ax.set_ylabel('Index')
ax.set_xlabel('')
plt.show()

rough

Zoom in on real bottles: the top 2 best-selling 750ml products in each group. Jim Beam (American, no tariff) is the only one with a price increase.

In [ ]:
top = con.sql("""
    WITH ranked AS (
        SELECT origin, item_no, product,
               SUM(bottles) AS total_bottles,
               ROW_NUMBER() OVER (PARTITION BY origin ORDER BY SUM(bottles) DESC) AS rank
        FROM whiskey_monthly
        WHERE bottle_volume_ml = 750
        GROUP BY origin, item_no, product
    )
    SELECT w.month, r.origin, r.product,
           w.calc_price, w.list_price, w.supplier_cost, w.bottles
    FROM whiskey_monthly w
    JOIN ranked r ON w.item_no = r.item_no
    WHERE r.rank <= 2
    ORDER BY r.origin, r.product, w.month
""").df()

products = top['product'].unique()
fig, axes = plt.subplots(len(products), 1, figsize=(12, 3 * len(products)), sharex=True)
for ax, name in zip(axes, products):
    d = top[top['product'] == name]
    ax.plot(d['month'], d['calc_price'], marker='o', label='paid (dollars / bottles)')
    ax.plot(d['month'], d['list_price'], marker='s', label='official list price')
    ax.set_title(f"{d['origin'].iloc[0]}: {name}")
    ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

Hennessy VS changed item numbers across most sizes in 2024 to 2025. Prices stayed the same under the new codes, so no price increase is hiding behind them.

In [ ]:
con.sql("""
    SELECT item_no, product, bottle_volume_ml,
           MIN(month) AS first_month,
           MAX(month) AS last_month,
           ROUND(MEDIAN(calc_price), 2) AS typical_price,
           ROUND(SUM(bottles)) AS bottles
    FROM whiskey_monthly
    WHERE product ILIKE '%HENNESSY VS%'
    GROUP BY item_no, product, bottle_volume_ml
    ORDER BY bottle_volume_ml, first_month
""").df()

## Step 6: If prices did not rise, why did sales fall?

Availability: how many different products were sold, and how many stores ordered each group.

In [ ]:
con.sql("""
    WITH tagged AS (
        SELECT origin, item_no,
               CASE
                   WHEN month BETWEEN '2024-04-01' AND '2024-10-01' THEN 'y2024'
                   WHEN month BETWEEN '2025-04-01' AND '2025-10-01' THEN 'y2025'
                   WHEN month BETWEEN '2026-04-01' AND '2026-10-01' THEN 'y2026'
               END AS period
        FROM whiskey_monthly
        WHERE bottles > 0
    ),
    counts AS (
        SELECT origin,
               COUNT(DISTINCT CASE WHEN period = 'y2024' THEN item_no END) AS products_2024,
               COUNT(DISTINCT CASE WHEN period = 'y2025' THEN item_no END) AS products_2025,
               COUNT(DISTINCT CASE WHEN period = 'y2026' THEN item_no END) AS products_2026
        FROM tagged
        GROUP BY origin
    )
    SELECT origin, products_2024, products_2025, products_2026,
           ROUND(100.0 * (products_2025 / products_2024 - 1), 1) AS pct_change_2025,
           ROUND(100.0 * (products_2026 / products_2024 - 1), 1) AS pct_change_2026
    FROM counts
    ORDER BY origin
""").df()

In [ ]:
con.sql("""
    WITH tagged AS (
        SELECT g.origin, s.store_no,
               CASE
                   WHEN s.ordered_on BETWEEN '2024-04-01' AND '2024-10-31' THEN 'y2024'
                   WHEN s.ordered_on BETWEEN '2025-04-01' AND '2025-10-31' THEN 'y2025'
                   WHEN s.ordered_on BETWEEN '2026-04-01' AND '2026-10-31' THEN 'y2026'
               END AS period
        FROM sales s
        JOIN category_groups g ON s.category_name = g.category_name
        WHERE s.sales_bottles > 0
    ),
    counts AS (
        SELECT origin,
               COUNT(DISTINCT CASE WHEN period = 'y2024' THEN store_no END) AS stores_2024,
               COUNT(DISTINCT CASE WHEN period = 'y2025' THEN store_no END) AS stores_2025,
               COUNT(DISTINCT CASE WHEN period = 'y2026' THEN store_no END) AS stores_2026
        FROM tagged
        GROUP BY origin
    )
    SELECT origin, stores_2024, stores_2025, stores_2026,
           ROUND(100.0 * (stores_2025 / stores_2024 - 1), 1) AS pct_change_2025,
           ROUND(100.0 * (stores_2026 / stores_2024 - 1), 1) AS pct_change_2026
    FROM counts
    ORDER BY origin
""").df()

What kind of products disappeared (sold in mid-2024, never sold again from April 2025 to October 2026)? Slow sellers in every group, but the EU lost more of them (35% vs 27% for American).

In [ ]:
con.sql("""
    WITH per_product AS (
        SELECT origin, item_no,
               SUM(CASE WHEN month BETWEEN '2024-04-01' AND '2024-10-01' THEN bottles ELSE 0 END) AS bottles_2024,
               SUM(CASE WHEN month BETWEEN '2025-04-01' AND '2026-10-01' THEN bottles ELSE 0 END) AS bottles_after,
               MEDIAN(calc_price) AS price
        FROM whiskey_monthly
        GROUP BY origin, item_no
    )
    SELECT origin,
           CASE WHEN bottles_after > 0 THEN 'still sold' ELSE 'disappeared' END AS status,
           COUNT(*) AS products,
           ROUND(MEDIAN(bottles_2024)) AS typical_bottles_2024,
           ROUND(MEDIAN(price), 2) AS typical_price
    FROM per_product
    WHERE bottles_2024 > 0
    GROUP BY origin, status
    ORDER BY origin, status
""").df()

## Step 7: The bridge (price, range and continuing products)

Split each group's change in bottles (vs April to October 2024) into products lost, products added, and the change in products sold in both periods. `lost + new` together is the range effect (item-number changes, like Hennessy's, cancel out there). The totals must match Step 4.

In [ ]:
def bridge(after_start, after_end, label):
    return con.sql(f"""
        WITH per_product AS (
            SELECT origin, item_no,
                   SUM(CASE WHEN month BETWEEN '2024-04-01' AND '2024-10-01'
                            THEN bottles ELSE 0 END) AS before,
                   SUM(CASE WHEN month BETWEEN '{after_start}' AND '{after_end}'
                            THEN bottles ELSE 0 END) AS after
            FROM whiskey_monthly
            GROUP BY origin, item_no
        ),
        classified AS (
            SELECT origin, before, after,
                   CASE WHEN before > 0 AND after > 0 THEN 'continuing'
                        WHEN before > 0 THEN 'lost'
                        WHEN after  > 0 THEN 'new'
                   END AS status
            FROM per_product
        )
        SELECT '{label}' AS comparison,
               origin,
               ROUND(100.0 * SUM(CASE WHEN status = 'lost' THEN -before ELSE 0 END) / SUM(before), 1) AS lost_pct,
               ROUND(100.0 * SUM(CASE WHEN status = 'new' THEN after ELSE 0 END) / SUM(before), 1) AS new_pct,
               ROUND(100.0 * SUM(CASE WHEN status = 'continuing' THEN after - before ELSE 0 END) / SUM(before), 1) AS continuing_pct,
               ROUND(100.0 * (SUM(after) - SUM(before)) / SUM(before), 1) AS total_pct
        FROM classified
        WHERE status IS NOT NULL
        GROUP BY origin
        ORDER BY origin
    """).df()

result = pd.concat([
    bridge('2025-04-01', '2025-10-01', '2025 vs 2024'),
    bridge('2026-04-01', '2026-10-01', '2026 vs 2024'),
])
result

## Step 8: Is the gap real or chance?

Monthly gap between each import group and American (log ratio x 100), before vs after April 2025, Welch t-test.

Result: EU shift -10.0 points, p = 0.0016 (significant). UK shift -5.8 points, p = 0.065 (not significant at 5%; the smallest, noisiest group). The simple test ignores month-to-month correlation, so the p-values are somewhat optimistic.

In [ ]:
gap = con.sql("""
    WITH monthly AS (
        SELECT month, origin, SUM(bottles) AS bottles
        FROM whiskey_monthly
        GROUP BY month, origin
    ),
    side_by_side AS (
        SELECT month,
               SUM(CASE WHEN origin = 'American' THEN bottles END) AS american,
               SUM(CASE WHEN origin = 'EU' THEN bottles END) AS eu,
               SUM(CASE WHEN origin = 'UK' THEN bottles END) AS uk
        FROM monthly
        GROUP BY month
    )
    SELECT month,
           100 * LN(eu / american) AS eu_gap,
           100 * LN(uk / american) AS uk_gap,
           CASE WHEN month < '2025-04-01' THEN 'before' ELSE 'after' END AS period
    FROM side_by_side
    ORDER BY month
""").df()

for col, name in [('eu_gap', 'EU'), ('uk_gap', 'UK')]:
    before = gap[gap['period'] == 'before'][col]
    after  = gap[gap['period'] == 'after'][col]
    shift = after.mean() - before.mean()
    wobble = before.std()
    t, p = stats.ttest_ind(after, before, equal_var=False)
    print(f"{name}: shift = {shift:.1f} points | normal wobble before = +/-{wobble:.1f} points | p-value = {p:.4f}")